# mIF Pipeline Stage Notebook: Setup, Merge, InstanSeg, Nimbus

Use this notebook in the `instanseg_nimbus` environment.

It covers the artifact-producing stages:
- `setup_slide(...)`
- `merge_slide_ometiffs(...)`
- `run_instanseg(...)`
- optional `run_nimbus_chunked(...)`

The outputs from this notebook are the handoff artifacts for final SpatialData assembly in the separate `harpy` environment.


In [1]:
from pathlib import Path
import json
import platform
import traceback

import mif_pipeline
from mif_pipeline import (
    load_config,
    merge_slide_ometiffs,
    run_instanseg,
    run_nimbus_chunked,
    run_nimbus_multislide,
    setup_slide,
)
from mif_pipeline.config import get_slide_config, resolve_nimbus_multislide_inputs

print("python:", platform.python_version())
print("mif_pipeline:", getattr(mif_pipeline, "__file__", "unknown"))


python: 3.10.20
mif_pipeline: /home/ratnayn/codex/mIF-pipeline/src/mif_pipeline/__init__.py


In [2]:
# Edit these for your current run.
CONFIG_PATH = Path("~/codex/mIF-pipeline/prototyping/prototype_v2-Crop.yaml").expanduser()
SLIDE_ID = "SLIDE-0329_crop_2048"

# Optional: limit Nimbus to specific chunk indices, for example [0] or [0, 1].
NIMBUS_CHUNK_INDICES = None

FORCE = False
RUN_SETUP = True
RUN_MERGE = True
RUN_INSTANSEG = True
RUN_NIMBUS = True

SETUP_DRY_RUN = not RUN_SETUP
MERGE_DRY_RUN = not RUN_MERGE
INSTANSEG_DRY_RUN = not RUN_INSTANSEG
NIMBUS_DRY_RUN = not RUN_NIMBUS


In [3]:
def show(value):
    print(json.dumps(value, indent=2, default=str))


def stage_call(label, func, *args, **kwargs):
    print(f"\n=== {label} ===")
    try:
        result = func(*args, **kwargs)
        show(result)
        return result
    except Exception as exc:
        print(f"{type(exc).__name__}: {exc}")
        traceback.print_exc()
        return None


In [4]:
print("config path:", CONFIG_PATH)
print("config exists:", CONFIG_PATH.exists())

config = load_config(CONFIG_PATH)
raw_slide = config["slides"][SLIDE_ID]
top_level_nimbus = config.get("nimbus") or {}
multislide_block = top_level_nimbus.get("multislide") or {}
multislide_enabled = bool(multislide_block.get("enabled", False))
runtime_slide_ids = list(multislide_block.get("slide_ids") or [SLIDE_ID]) if multislide_enabled else [SLIDE_ID]

print("slide_id:", SLIDE_ID)
print("nimbus_multislide_enabled:", multislide_enabled)
print("runtime_slide_ids:", runtime_slide_ids)

show({
    "top_level_setup": config.get("setup"),
    "top_level_full_merge": config.get("full_merge"),
    "top_level_instanseg": config.get("instanseg"),
    "top_level_nimbus": config.get("nimbus"),
    "slide_block": raw_slide,
})


config path: /home/ratnayn/codex/mIF-pipeline/prototyping/prototype_v2-Crop.yaml
config exists: True
slide_id: SLIDE-0329_crop_2048
nimbus_multislide_enabled: True
runtime_slide_ids: ['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2']
{
  "top_level_setup": {
    "channel_patterns": [
      "*.tif",
      "*.tiff",
      "*.ome.tif",
      "*.ome.tiff"
    ],
    "channel_map_output": "channel_map.generated.json",
    "include_round_in_alias": true
  },
  "top_level_full_merge": {
    "enabled": true,
    "exclude_channels": [],
    "suffix": "_full.ome.tif",
    "compression": "zlib",
    "tile": [
      256,
      256
    ],
    "bigtiff": true
  },
  "top_level_instanseg": {
    "channels": [
      "R1_DAPI",
      "R1_FITC_AF",
      "R4_P19_POLYRAT",
      "R4_ANXA10_647",
      "R4_LGALS4_750"
    ],
    "model": "fluorescence_nuclei_and_cells",
    "mode": "medium",
    "prediction_tag": "_instanseg_prediction",
    "tile_size": 1000,
    "overlap": 100,
    "batch_size": 1,
    

## Setup / Channel Map Generation

This stage generates or validates the channel map from the configured `slide_dir` and setup patterns.


In [5]:
setup_results = {}
for target_slide_id in runtime_slide_ids:
    setup_results[target_slide_id] = stage_call(
        f"setup_slide({target_slide_id}, dry_run={SETUP_DRY_RUN})",
        setup_slide,
        config,
        target_slide_id,
        force=FORCE,
        dry_run=SETUP_DRY_RUN,
    )



=== setup_slide(SLIDE-0329_crop_2048, dry_run=False) ===
{
  "slide_id": "SLIDE-0329_crop_2048",
  "slide_dir": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048",
  "channel_map_output": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/channel_map.generated.json",
  "channel_patterns": [
    "*.tif",
    "*.tiff",
    "*.ome.tif",
    "*.ome.tiff"
  ],
  "include_round_in_alias": true,
  "channel_count": 24,
  "aliases": [
    "R1_CY3_AF",
    "R1_CY5_AF",
    "R1_CY7_AF",
    "R1_FITC_AF",
    "R1_BIT2_RS0584_CY3B",
    "R1_BIT3_RS0015_CY5",
    "R1_BIT4_RS0083_750",
    "R1_DAPI",
    "R1_BIT1_RS0996_488",
    "R2_BIT6_RS0639_CY3B",
    "R2_BIT7_RS0109_CY5",
    "R2_BIT8_RS0255_750",
    "R2_DAPI",
    "R2_BIT5_RS1047_488",
    "R3_BIT10_RS0763_CY3B",
    "R3_BIT11_RS1312_CY5",
    "R3_BIT12_RS0237_750",
    "R3_DAPI",
    "R3_BIT9_RS0805_488",
    "R4_P19_POLYRAT",
    "R4_ANXA10_647",
    "R4_LGALS4_750",
    "R4_DAPI",
    "R4_GFP_POLY_AF488"
  ],
  "stat

In [6]:
runtime_slides = {target_slide_id: get_slide_config(config, target_slide_id) for target_slide_id in runtime_slide_ids}
all_channel_maps_exist = True

for target_slide_id in runtime_slide_ids:
    target_slide = runtime_slides[target_slide_id]
    channel_map_path = Path(target_slide["channel_map_file"])
    print(f"\n=== channel map: {target_slide_id} ===")
    print("channel_map_file:", channel_map_path)
    print("channel map exists:", channel_map_path.exists())

    if channel_map_path.exists():
        channel_map = json.loads(channel_map_path.read_text())
        print("channel count:", len(channel_map))
        show(channel_map[: min(10, len(channel_map))])
    else:
        all_channel_maps_exist = False
        print("Run setup with RUN_SETUP = True to write the generated channel map to disk.")

if multislide_enabled and all_channel_maps_exist:
    multislide_inputs = resolve_nimbus_multislide_inputs(config, runtime_slide_ids)
    print("\nresolved multislide inputs:")
    show(multislide_inputs)
else:
    multislide_inputs = None
    if multislide_enabled:
        print("\nMultislide inputs will resolve after channel maps exist.")



=== channel map: SLIDE-0329_crop_2048 ===
channel_map_file: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/channel_map.generated.json
channel map exists: True
channel count: 24
[
  {
    "alias": "R1_CY3_AF",
    "path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/SLIDE-0329_1.0.1_R000_Cy3_AF_I.ome.tif",
    "nimbus_name": "SLIDE-0329_1.0.1_R000_Cy3_AF_I"
  },
  {
    "alias": "R1_CY5_AF",
    "path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/SLIDE-0329_1.0.1_R000_Cy5_AF_I.ome.tif",
    "nimbus_name": "SLIDE-0329_1.0.1_R000_Cy5_AF_I"
  },
  {
    "alias": "R1_CY7_AF",
    "path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/SLIDE-0329_1.0.1_R000_Cy7_AF_I.ome.tif",
    "nimbus_name": "SLIDE-0329_1.0.1_R000_Cy7_AF_I"
  },
  {
    "alias": "R1_FITC_AF",
    "path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/SLIDE-0329_1.0.1_R000_FITC_AF_I.ome.tif",
    "nimbus_name": "SLIDE-0329_1.0.1_R000_FITC_AF_I"
  },
  {
    "alias": "R1_BI

## Merge Stage

This writes the canonical merged image artifact:
- `full_merge.ome.tif` for InstanSeg, Nimbus, and final SpatialData assembly


In [7]:
merge_results = {}
for target_slide_id in runtime_slide_ids:
    merge_results[target_slide_id] = stage_call(
        f"merge_slide_ometiffs({target_slide_id}, dry_run={MERGE_DRY_RUN})",
        merge_slide_ometiffs,
        config,
        target_slide_id,
        force=FORCE,
        dry_run=MERGE_DRY_RUN,
    )



=== merge_slide_ometiffs(SLIDE-0329_crop_2048, dry_run=False) ===
{
  "slide_id": "SLIDE-0329_crop_2048",
  "slide_dir": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048",
  "outputs": {
    "full_merge": {
      "status": "skipped",
      "ome_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif",
      "channels": [
        "R1_CY3_AF",
        "R1_CY5_AF",
        "R1_CY7_AF",
        "R1_FITC_AF",
        "R1_BIT2_RS0584_CY3B",
        "R1_BIT3_RS0015_CY5",
        "R1_BIT4_RS0083_750",
        "R1_DAPI",
        "R1_BIT1_RS0996_488",
        "R2_BIT6_RS0639_CY3B",
        "R2_BIT7_RS0109_CY5",
        "R2_BIT8_RS0255_750",
        "R2_DAPI",
        "R2_BIT5_RS1047_488",
        "R3_BIT10_RS0763_CY3B",
        "R3_BIT11_RS1312_CY5",
        "R3_BIT12_RS0237_750",
        "R3_DAPI",
        "R3_BIT9_RS0805_488",
        "R4_P19_POLYRAT",
        "R4_ANXA10_647",
        "R4_LGALS4_750",
        "R4_DAPI",
        "R4_GF

## InstanSeg / Mask Export

This stage consumes `full_merge.ome.tif`, selects the aliases listed in `instanseg.channels`, and writes the whole-cell and nuclear label TIFFs that will later be imported as SpatialData labels.

When multislide Nimbus is enabled, we still run merge and InstanSeg per slide before the shared Nimbus call.


In [8]:
instanseg_results = {}
for target_slide_id in runtime_slide_ids:
    instanseg_results[target_slide_id] = stage_call(
        f"run_instanseg({target_slide_id}, dry_run={INSTANSEG_DRY_RUN})",
        run_instanseg,
        config,
        target_slide_id,
        force=FORCE,
        dry_run=INSTANSEG_DRY_RUN,
    )



=== run_instanseg(SLIDE-0329_crop_2048, dry_run=False) ===
[instanseg] skipping SLIDE-0329_crop_2048: mask outputs already exist at /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks (force=False)
{
  "slide_id": "SLIDE-0329_crop_2048",
  "ome_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/SLIDE-0329_crop_2048_full.ome.tif",
  "channels": [
    "R1_DAPI",
    "R1_FITC_AF",
    "R4_P19_POLYRAT",
    "R4_ANXA10_647",
    "R4_LGALS4_750"
  ],
  "channel_indices": [
    7,
    3,
    19,
    20,
    21
  ],
  "mode": "medium",
  "model": "fluorescence_nuclei_and_cells",
  "tile_size": 1000,
  "overlap": 100,
  "batch_size": 1,
  "pixel_size_um": 0.325,
  "prediction_tag": "_instanseg_prediction",
  "cell_mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_whole_cell.tiff",
  "nuclear_mask_path": "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-032

In [9]:
for target_slide_id, instanseg_result in instanseg_results.items():
    print(f"\n=== masks: {target_slide_id} ===")
    if instanseg_result is not None:
        print("cell_mask_path:", instanseg_result.get("cell_mask_path"))
        print("nuclear_mask_path:", instanseg_result.get("nuclear_mask_path"))
        print("mask_dir:", instanseg_result.get("mask_dir"))



=== masks: SLIDE-0329_crop_2048 ===
cell_mask_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_whole_cell.tiff
nuclear_mask_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks/SLIDE-0329_crop_2048_nuclear.tiff
mask_dir: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/pipeline_outputs/masks

=== masks: SLIDE-0329_crop_2048_2 ===
cell_mask_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs/masks/SLIDE-0329_crop_2048_2_whole_cell.tiff
nuclear_mask_path: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs/masks/SLIDE-0329_crop_2048_2_nuclear.tiff
mask_dir: /mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2/pipeline_outputs/masks


## Optional Nimbus Stage

Nimbus is optional.

- If `nimbus.multislide.enabled: false`, this notebook calls `run_nimbus_chunked(...)` for the current slide.
- If `nimbus.multislide.enabled: true`, this notebook calls `run_nimbus_multislide(...)` using the YAML `slide_ids` (or the override list above) and writes per-slide tables under the multislide output root.


In [10]:
primary_slide = runtime_slides[SLIDE_ID]
nimbus_enabled = bool((primary_slide.get("nimbus") or {}).get("enabled", False))
multislide_enabled = bool((((config.get("nimbus") or {}).get("multislide") or {}).get("enabled", False)))
print("Nimbus enabled:", nimbus_enabled)
print("Nimbus multislide enabled:", multislide_enabled)

if nimbus_enabled:
    if multislide_enabled:
        selected_slide_ids = list(runtime_slide_ids)
        print("Nimbus multislide slide_ids:", selected_slide_ids)
        print("Nimbus chunk_indices:", NIMBUS_CHUNK_INDICES)
        nimbus_result = stage_call(
            f"run_nimbus_multislide({selected_slide_ids}, dry_run={NIMBUS_DRY_RUN})",
            run_nimbus_multislide,
            config,
            selected_slide_ids,
            chunk_indices=NIMBUS_CHUNK_INDICES,
            force=FORCE,
            dry_run=NIMBUS_DRY_RUN,
        )
    else:
        print("Nimbus chunk_indices:", NIMBUS_CHUNK_INDICES)
        nimbus_result = stage_call(
            f"run_nimbus_chunked({SLIDE_ID}, dry_run={NIMBUS_DRY_RUN})",
            run_nimbus_chunked,
            config,
            SLIDE_ID,
            chunk_indices=NIMBUS_CHUNK_INDICES,
            force=FORCE,
            dry_run=NIMBUS_DRY_RUN,
        )
else:
    print("Nimbus is disabled in the config.")
    nimbus_result = None


Nimbus enabled: True
Nimbus multislide enabled: True
Nimbus multislide slide_ids: ['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2']
Nimbus chunk_indices: None

=== run_nimbus_multislide(['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2'], dry_run=False) ===


/home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/utils.py:10: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Detected single-page OME-TIFF channel files. Each file contains 1 page.
All inputs are valid
Dataset initialized with
FOVs: ['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2'].
Channels: ['R1_DAPI'].
Iterate over fovs...
Checking for updated model checkpoints on HuggingFace Hub...
Using existing checkpoint: /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
Loaded weights from /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
All inputs are valid.
Available GPUs:  1
Predictions will be saved in /mnt/c/analysis/Data_prototype/crop_nimbus_multislide/chunk_000
Iterating through fovs will take a while...
Predicting /mnt/c/analysis/Data_prototype/crop_nimbus_multislide/_multislide_fovs/SLIDE-0329_crop_2048...


  0%|          | 0/1 [00:00<?, ?it/s]

Predicting /mnt/c/analysis/Data_prototype/crop_nimbus_multislide/_multislide_fovs/SLIDE-0329_crop_2048_2...


...

Detected single-page OME-TIFF channel files. Each file contains 1 page.
All inputs are valid
Dataset initialized with
FOVs: ['SLIDE-0329_crop_2048', 'SLIDE-0329_crop_2048_2'].
Channels: ['R4_ANXA10_647'].
Iterate over fovs...
Checking for updated model checkpoints on HuggingFace Hub...
Using existing checkpoint: /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
Loaded weights from /home/ratnayn/miniconda3/envs/instanseg_nimbus/lib/python3.10/site-packages/nimbus_inference/assets/V1.pt
All inputs are valid.
Available GPUs:  1
Predictions will be saved in /mnt/c/analysis/Data_prototype/crop_nimbus_multislide/chunk_003
Iterating through fovs will take a while...
Predicting /mnt/c/analysis/Data_prototype/crop_nimbus_multislide/_multislide_fovs/SLIDE-0329_crop_2048...


  0%|          | 0/1 [00:00<?, ?it/s]

Predicting /mnt/c/analysis/Data_prototype/crop_nimbus_multislide/_multislide_fovs/SLIDE-0329_crop_2048_2...


  0%|          | 0/1 [00:00<?, ?it/s]

{
  "slide_ids": [
    "SLIDE-0329_crop_2048",
    "SLIDE-0329_crop_2048_2"
  ],
  "mode": "multislide",
  "fov_paths": [
    "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048",
    "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048_2"
  ],
  "fov_name_to_slide": {
    "SLIDE-0329_crop_2048": "SLIDE-0329_crop_2048",
    "SLIDE-0329_crop_2048_2": "SLIDE-0329_crop_2048_2"
  },
  "raw_paths_by_slide": {
    "SLIDE-0329_crop_2048": [
      "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/SLIDE-0329_1.0.4_R000_DAPI__FINAL_F.ome.tif",
      "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/SLIDE-0329_4.0.4_R000_FITC_GFP-poly-AF488_FINAL_AFR_F.ome.tif",
      "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/SLIDE-0329_4.0.4_R000_Cy3_p19-polyRat_FINAL_AFR_F.ome.tif",
      "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048/SLIDE-0329_4.0.4_R000_Cy5_Anxa10-647_FINAL_AFR_F.ome.tif"
    ],
    "SLIDE-0329_crop_2048_2": [
      "/mnt/c/analysis/Data_prototype/SLIDE-0329_crop_2048

## Handoff Summary

At the end of this notebook, the expected handoff artifacts for the `harpy` notebook are:
- `full_merge.ome.tif`
- whole-cell label TIFF
- nuclear label TIFF
- optional Nimbus `cell_table_full.csv`
- if multislide Nimbus is enabled, the per-slide table under `nimbus.multislide.output_dir / per_slide / <slide_id> / cell_table_full.csv`
